# Section 01: Designing and Planning a Cloud Solution Architecture

**Companion notebook for**: `01-designing-cloud-solutions.html`

## Learning Objectives
- Explore GCP compute, storage, and networking options programmatically
- Compare services using architecture decision matrices
- Practice gcloud commands for solution design
- Work with Terraform configurations for landing zones
- Interact with Gemini and Agent Builder APIs

## Prerequisites
- A Google Cloud project with billing enabled
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q google-cloud-compute google-cloud-storage google-cloud-resource-manager vertexai

In [ ]:
import os
import json

PROJECT_ID = os.environ.get('GCP_PROJECT_ID', 'your-project-id')
REGION = 'us-central1'
ZONE = 'us-central1-a'

print(f'Project: {PROJECT_ID}')
print(f'Region:  {REGION}')
print(f'Zone:    {ZONE}')

if PROJECT_ID == 'your-project-id':
    print('\nWARNING: Set GCP_PROJECT_ID or replace the value above.')

In [ ]:
# Authenticate in Colab
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab.')
except ImportError:
    print('Not in Colab -- assuming local gcloud auth.')

---
## Section 1: Compute Option Comparison

As a Cloud Architect, you must quickly compare compute options based on workload requirements.

In [ ]:
compute_options = [
    {'service': 'Compute Engine', 'abstraction': 'IaaS (VMs)', 'scale_to_zero': False, 'stateful': True, 'gpu': True, 'best_for': 'Legacy apps, Windows, HPC'},
    {'service': 'GKE', 'abstraction': 'CaaS (Containers)', 'scale_to_zero': False, 'stateful': True, 'gpu': True, 'best_for': 'Microservices, service mesh'},
    {'service': 'Cloud Run', 'abstraction': 'Serverless containers', 'scale_to_zero': True, 'stateful': False, 'gpu': False, 'best_for': 'HTTP APIs, event-driven'},
    {'service': 'App Engine', 'abstraction': 'PaaS', 'scale_to_zero': False, 'stateful': False, 'gpu': False, 'best_for': 'Web apps, prototyping'},
    {'service': 'Cloud Functions', 'abstraction': 'FaaS', 'scale_to_zero': True, 'stateful': False, 'gpu': False, 'best_for': 'Event handlers, webhooks'},
]

print(f"{'Service':<20} {'Abstraction':<25} {'Scale-0':<10} {'Stateful':<10} {'GPU':<6} {'Best For'}")
print('-' * 105)
for o in compute_options:
    print(f"{o['service']:<20} {o['abstraction']:<25} {'Yes' if o['scale_to_zero'] else 'No':<10} {'Yes' if o['stateful'] else 'No':<10} {'Yes' if o['gpu'] else 'No':<6} {o['best_for']}")

---
## Section 2: Storage Decision Framework

Build a storage selection helper that recommends the right GCP storage service.

In [ ]:
def recommend_storage(data_model, size_tb, global_reach=False):
    if data_model == 'unstructured':
        return 'Cloud Storage', 'Object storage for blobs, media, backups'
    if data_model == 'relational':
        if global_reach:
            return 'Cloud Spanner', 'Global strong consistency with SQL'
        elif size_tb < 10:
            return 'Cloud SQL', 'Managed MySQL/PostgreSQL for OLTP < 10TB'
        else:
            return 'Cloud Spanner', 'Horizontal scaling beyond Cloud SQL'
    if data_model == 'document':
        return 'Firestore', 'Document DB for mobile/web'
    if data_model == 'wide-column' and size_tb >= 1:
        return 'Bigtable', 'Wide-column for IoT, time-series, >1TB'
    if data_model == 'analytics':
        return 'BigQuery', 'Serverless data warehouse for OLAP'
    if data_model == 'cache':
        return 'Memorystore', 'In-memory Redis/Memcached'
    return 'Cloud Storage', 'Default'

scenarios = [
    ('relational', 5, True), ('relational', 3, False), ('unstructured', 100, False),
    ('wide-column', 50, False), ('document', 0.5, False), ('analytics', 500, False),
]
print(f"{'Model':<15} {'Size(TB)':<10} {'Global':<8} {'Recommendation'}")
print('-' * 75)
for dm, sz, gl in scenarios:
    svc, reason = recommend_storage(dm, sz, gl)
    print(f"{dm:<15} {sz:<10} {'Yes' if gl else 'No':<8} {svc} -- {reason}")

---
## Section 3: Network Design Commands

Practice VPC creation and configuration gcloud commands (display only).

In [ ]:
commands = [
    ('Create custom VPC', 'gcloud compute networks create prod-vpc --subnet-mode=custom'),
    ('Create web subnet', 'gcloud compute networks subnets create web-subnet --network=prod-vpc --region=us-central1 --range=10.0.1.0/24 --enable-private-ip-google-access'),
    ('Allow HTTP', 'gcloud compute firewall-rules create allow-http --network=prod-vpc --allow=tcp:80,tcp:443 --source-ranges=0.0.0.0/0 --target-tags=http-server'),
    ('Allow internal', 'gcloud compute firewall-rules create allow-internal --network=prod-vpc --allow=tcp,udp,icmp --source-ranges=10.0.0.0/16'),
    ('Create Cloud Router', 'gcloud compute routers create nat-router --network=prod-vpc --region=us-central1'),
    ('Create Cloud NAT', 'gcloud compute routers nats create prod-nat --router=nat-router --region=us-central1 --nat-all-subnet-ip-ranges --auto-allocate-nat-external-ips'),
]
for desc, cmd in commands:
    print(f'## {desc}')
    print(f'   {cmd}\n')

---
## Section 4: Migration Strategy Selector

In [ ]:
def recommend_migration(readiness, timeline_months, app_type):
    if app_type == 'saas_replacement': return 'Repurchase'
    if readiness == 'none' and timeline_months <= 3: return 'Rehost (Lift & Shift)'
    if readiness == 'containerized': return 'Replatform'
    if readiness == 'cloud-native': return 'Refactor'
    return 'Rehost'

workloads = [
    ('Java monolith', 'none', 2, 'legacy'), ('Docker microservices', 'containerized', 6, 'web'),
    ('On-prem MySQL', 'none', 4, 'database'), ('Email server', 'none', 1, 'saas_replacement'),
    ('Python ML pipeline', 'cloud-native', 6, 'ml'),
]
print(f"{'Workload':<25} {'Strategy'}")
print('-' * 50)
for name, readiness, timeline, atype in workloads:
    strategy = recommend_migration(readiness, timeline, atype)
    print(f'{name:<25} {strategy}')

---
## Section 5: Terraform Landing Zone

In [ ]:
tf_config = '''# main.tf -- GCP Landing Zone
terraform {
  backend "gcs" {
    bucket = "my-org-tf-state"
    prefix = "landing-zone"
  }
  required_providers {
    google = { source = "hashicorp/google", version = "~> 5.0" }
  }
}

module "project" {
  source          = "terraform-google-modules/project-factory/google"
  version         = "~> 15.0"
  name            = "prod-workload"
  org_id          = var.org_id
  folder_id       = var.folder_id
  billing_account = var.billing_account
  activate_apis   = [
    "compute.googleapis.com",
    "container.googleapis.com",
    "sqladmin.googleapis.com",
    "monitoring.googleapis.com",
  ]
}

module "vpc" {
  source       = "terraform-google-modules/network/google"
  version      = "~> 9.0"
  project_id   = module.project.project_id
  network_name = "prod-vpc"
  subnets = [
    { subnet_name = "web", subnet_ip = "10.0.1.0/24", subnet_region = "us-central1" },
    { subnet_name = "db",  subnet_ip = "10.0.2.0/24", subnet_region = "us-central1" },
  ]
}
'''
print(tf_config)

---
## Section 6: Gemini for Architecture Assistance

In [ ]:
try:
    import vertexai
    from vertexai.generative_models import GenerativeModel, GenerationConfig
    vertexai.init(project=PROJECT_ID, location=REGION)
    model = GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(
        'You are a GCP architect. Recommend services for: 1) global web app, 2) 10TB relational DB with strong consistency, 3) real-time event processing from 500 stores. Be concise, table format.',
        generation_config=GenerationConfig(temperature=0.2, max_output_tokens=500)
    )
    print(response.text)
except Exception as e:
    print(f'Gemini call failed: {e}')
    print('\nMock recommendation:')
    print('| Requirement | Service | Why |')
    print('|---|---|---|')
    print('| Global web app | Cloud Run + Global LB + CDN | Serverless, edge caching |')
    print('| 10TB relational | Cloud Spanner multi-region | Strong consistency, SQL |')
    print('| Real-time events | Pub/Sub + Dataflow | Managed streaming |')

---
## Section 7: Well-Architected Framework Checklist

In [ ]:
pillars = {
    'Operational Excellence': ['IaC (Terraform)', 'CI/CD pipelines', 'Monitoring + alerting', 'Runbooks', 'Autoscaling'],
    'Security': ['Least-privilege IAM', 'CMEK encryption', 'VPC-SC', 'Audit logging', 'Org policies'],
    'Reliability': ['Multi-zone deployment', 'Health checks', 'DR plan tested', 'SLOs defined', 'Error budgets'],
    'Cost Optimization': ['Right-sizing (Recommender)', 'CUDs for steady workloads', 'Storage lifecycle', 'Billing alerts', 'Spot VMs'],
    'Performance': ['CDN for static assets', 'DB indexing/partitioning', 'Caching (Memorystore)', 'Load testing', 'Right machine types'],
}
for pillar, items in pillars.items():
    print(f'\n{pillar}')
    print('=' * 40)
    for i, item in enumerate(items, 1):
        print(f'  [ ] {i}. {item}')

---
## Summary

This notebook covered PCA Section 01 core patterns:

1. **Compute Selection** -- Decision matrix for GCE, GKE, Cloud Run, App Engine, Functions
2. **Storage Selection** -- Recommendation engine based on data model and scale
3. **Network Design** -- VPC, subnets, firewalls, NAT via gcloud
4. **Migration Strategy** -- The 6 Rs applied to workloads
5. **Terraform Landing Zone** -- Project factory and VPC modules
6. **Gemini Architecture** -- AI-assisted design decisions
7. **Well-Architected Checklist** -- Five pillars review

**Next**: Section 02 covers managing and provisioning cloud infrastructure.